In [0]:
df = spark.table("novacart_catalog.001_bronze.products")
df.display()

In [0]:
from pyspark.sql.functions import col, when, trim, regexp_replace, concat_ws, split,to_date,try_to_date
from pyspark.sql import functions as F

In [0]:
df = spark.table("novacart_catalog.001_bronze.products")
df.display()

In [0]:
 
df = df.withColumn("product_id", F.trim(F.col("product_id"))) \
       .withColumn("product_name", F.trim(F.col("product_name"))) \
       .withColumn("category", F.trim(F.col("category"))) \
       .withColumn("currency", F.trim(F.col("currency"))) \
       .withColumn("country_code", F.trim(F.col("country_code")))
 
display(df)

In [0]:
null_values = ["", "null", "NULL", "\\N", "-", "?", ","]
df = df.withColumn(
    "product_name",
    F.when(
        (F.col("product_name").isin(null_values)) | (F.col("product_name").isNull()),
        "unknown"
    ).otherwise(F.col("product_name"))
)
df.display()


In [0]:
df = df.withColumn(
    "product_id",
    F.when(F.col("product_id").rlike("^PROD[0-9]+$"), F.col("product_id"))
     .otherwise(None)
)
 
df = df.withColumn(
    "price",
    F.when(F.col("price") > 0, F.col("price")).otherwise(None)
)
 
df.display()

In [0]:
df.write.format("delta") .mode("overwrite") .option("overwriteSchema", "true") .saveAsTable("novacart_catalog.002_silver.products")